In [1]:
import shapefile
import pandas as pd
import shapely.geometry
from shapely.geometry import Point, shape, Polygon
import json
import geopandas as gpd
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
import plotly.express as px
import os
import plotly
import numpy as np

In [2]:
def yield_rank(block_df):
    block_df['Norm N2 Used'] = block_df['Nitrogen Used (kg/ha)']/(np.linalg.norm(block_df['Nitrogen Used (kg/ha)']))
    block_df['Norm Water Used'] = block_df['Water Used (mm)']/(np.linalg.norm(block_df['Water Used (mm)']))
    block_df['Norm Yield'] = block_df['Unadjusted Yield (kg/ha))']/(np.linalg.norm(block_df['Unadjusted Yield (kg/ha))']))
    #This block data is for a single treatment across all of a state, how can each block, and the district it belongs to be ranked.
    block_df['a_rank'] = 0.5*block_df['Norm Yield']+0.25*block_df['Norm Water Used']+0.25*block_df['Norm N2 Used']
    return block_df

In [3]:
#populate the district name for a df that has unique long and lat
def populdistname(block_df, sf, tn_dist_json, tnm):
    block_df_unique_ll = block_df[['Center Long', 'Center Lat']].drop_duplicates()
    all_shapes = sf.shapes()
    block_df_unique_ll['dt_name']=0
    for index, row in block_df_unique_ll.iterrows():
        point = Point(float((row['Center Long'])), float ((row['Center Lat'])))
        oid = -99
        for ashape in all_shapes:
            if point.within(shape(ashape)):
                oid = ashape.oid
                break
        if oid > 0:
            containing_dist_name =  tn_dist_json["features"][oid]["properties"]["dtname"]
            block_df_unique_ll['dt_name'][index]=containing_dist_name
    block_df_unique_ll = block_df_unique_ll[block_df_unique_ll['dt_name'] != 0]   
    block_df_unique_ll['geometry']=0
    for index, row in block_df_unique_ll.iterrows():
        for i in range (len(tnm)):
            if ((block_df_unique_ll['dt_name'][index])==(tnm['dtname'][i])):
                  block_df_unique_ll['geometry'][index]=tnm['geometry'][i]  
    return block_df_unique_ll

In [4]:
#populate district name in the original df
def populdtnameoforigdf(block_df, block_df_unique_ll):
    block_df['dt_name']=0
    for ind in range (len(block_df)):
        for i, r in block_df_unique_ll.iterrows():
            if (block_df['Center Long'][ind] == block_df_unique_ll['Center Long'][i]) and (block_df['Center Lat'][ind] == block_df_unique_ll['Center Lat'][i]):
                block_df['dt_name'][ind]=block_df_unique_ll['dt_name'][i] 
                break
    block_df = block_df[block_df['dt_name'] != 0]            
    return block_df            

In [5]:
#calculate the average yield per district
def avg_dist (block_df, var, tnm):
    if var == 'yield':
        col_to_use = 'Unadjusted Yield (kg/ha))'
    elif var == 'co2':
        col_to_use = 'CO2 Emitted (kg/ha)'
    elif var == 'n2':
        col_to_use = 'Nitrogen Used (kg/ha)'
    elif var == 'water':
        col_to_use = 'Water Used (mm)'
    elif var== 'rank':
        col_to_use='a_rank'    
    block_df = block_df.astype({'Center Lat': float, 'Center Long': float})
    block_df.rename(columns={"Unadjusted Yield (kg/ha))": "Yield (kg/ha)", 
                           "dt_name": "District", "a_rank": "Sample Yield/Env Rank"}, inplace=True)
    block_df = block_df[['District', 'Yield (kg/ha)', 'CO2 Emitted (kg/ha)', 'Water Used (mm)', 
                     'Nitrogen Applied (kg/ha)', 'Nitrogen Leached (kg/ha)','Sample Yield/Env Rank']]
    avg_df = block_df.groupby(['District']).mean([col_to_use]).reset_index()
    avg_df['geometry']=0
    for index in range (len(avg_df)):
        for i in range (len(tnm)):
            if ((avg_df['District'][index])==(tnm['dtname'][i])):
                  avg_df['geometry'][index]=tnm['geometry'][i] 
    return avg_df  

In [6]:
#convert to geo df and display a map
def converttogeo(avg_df):
    geo_df = gpd.GeoDataFrame(avg_df, crs=4326)
    mp = geo_df.explore(column='Yield (kg/ha)',
                          cmap='RdYlGn',
                          marker_kwds=dict(radius=5, fill=True), 
                          tooltip_kwds=dict(labels=True),   
                          scheme = 'naturalbreaks',
                          legend=True,
                          k=10,
                          tooltip=False,
                          legend_kwds=dict(colorbar=False),
                          style_kwds=dict(color="black",weight=2, opacity=0.4))
    return mp

In [ ]:
#generate the map
#total time taken to generate the map: 2 mins 15 secs
with open ('TN_districts.json') as jf:
    tn_dist_json = json.load (jf)
sf= shapefile.Reader('TN_districts.shp')
block_df= pd.read_pickle('TN_Sorghum_BlockReport_df.pkl')
tnm = gpd.read_file('./Shap/tn_shp_gh/TAMIL NADU_DISTRICTS.shp')
block_df = yield_rank(block_df)
block_df_unique_ll = populdistname(block_df, sf, tn_dist_json, tnm)
block_df = populdtnameoforigdf(block_df, block_df_unique_ll)
avg_df = avg_df = avg_dist(block_df, 'yield', tnm)
mp = converttogeo(avg_df)
mp